# Практикум 9. Подбор гиперпараметров

Сопровождает интерактивную лабораторию №9 («Подбор гиперпараметров»). Тема курса: Grid Search и Random Search

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *

# Учебный корпус: 4 темы. noise - доля "посторонних" слов (чем больше, тем труднее).
df = make_corpus(n_per_class=150, noise=0.55, seed=0)
# Свой корпус вместо учебного: df = pd.read_csv("texts.csv")  # столбцы: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
import time
from scipy.stats import loguniform
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split

Xtr, Xte, ytr, yte = train_test_split(df.text, df.label, test_size=0.25, stratify=df.label, random_state=0)
pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))])

## 1. Grid Search: перебор сетки

In [ ]:
grid = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [1, 3], "clf__C": [0.01, 0.1, 1, 10, 100]}
t0 = time.time()
gs = GridSearchCV(pipe, grid, cv=3, scoring="f1_macro", n_jobs=-1).fit(Xtr, ytr)
print("Grid:", round(time.time() - t0, 1), "с | комбинаций:", len(gs.cv_results_["params"]))
print("Лучшие параметры:", gs.best_params_, "| CV:", round(gs.best_score_, 3))

## 2. Random Search: тот же бюджет проб, случайные комбинации

In [ ]:
dist = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [1, 2, 3, 5], "clf__C": loguniform(1e-2, 1e2)}
t0 = time.time()
rs = RandomizedSearchCV(pipe, dist, n_iter=10, cv=3, scoring="f1_macro", n_jobs=-1, random_state=0).fit(Xtr, ytr)
print("Random:", round(time.time() - t0, 1), "с | комбинаций:", 10)
print("Лучшие параметры:", rs.best_params_, "| CV:", round(rs.best_score_, 3))

In [ ]:
res = pd.DataFrame(gs.cv_results_)
res["C"] = res["param_clf__C"].astype(float)
plt.scatter(res["C"], res["mean_test_score"], label="grid")
r2 = pd.DataFrame(rs.cv_results_); r2["C"] = r2["param_clf__C"].astype(float)
plt.scatter(r2["C"], r2["mean_test_score"], marker="x", label="random")
plt.xscale("log"); plt.xlabel("C"); plt.ylabel("F1 macro (CV)"); plt.legend(); plt.title("Какие значения C проверил каждый поиск"); plt.show()

## 3. Честная оценка на нетронутом тесте

In [ ]:
from sklearn.metrics import f1_score
for name, s in [("grid", gs), ("random", rs)]:
    print(name, "test F1:", round(f1_score(yte, s.predict(Xte), average="macro"), 3))

## Задания

1. Увеличьте `n_iter` случайного поиска до 40. Как растёт лучшая CV-оценка?
2. Добавьте в поиск другие модели (например, `LinearSVC`) через список параметров `clf`.
3. Попробуйте `HalvingRandomSearchCV` (`from sklearn.experimental import enable_halving_search_cv`). Чем он отличается?